# Day E6 — 예측 모델과 검증: 판별, 보정, 시간적 외부 검증, 누수

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/E-ehr/day-E6.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/E-ehr/day-E6.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 무작위 분할 vs 시간 분할, 그리고 보정 표

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

rng = np.random.default_rng(6)
n = 12000
year = rng.integers(2013, 2021, n)
age = rng.normal(66, 14, n)
sofa = np.clip(rng.poisson(4.5, n), 0, 20)
lact_true = np.exp(0.3 + 0.07 * sofa + rng.normal(0, 0.3, n))
lact_meas = np.where(year >= 2018, lact_true * 1.25, lact_true)   # 측정법 변경: 같은 환자, 25% 높게
drift = -0.12 * (year - 2013)
died = rng.random(n) < 1 / (1 + np.exp(-(-3.4 + drift + 0.03 * (age - 66) + 0.28 * sofa + 0.6 * np.log(lact_true))))
df = pd.DataFrame({"year": year, "age": age, "sofa": sofa, "log_lact": np.log(lact_meas), "died": died.astype(int)})
X = ["age", "sofa", "log_lact"]

def calib(y, p):
    lp = np.log(p / (1 - p))
    slope = sm.Logit(y, sm.add_constant(lp)).fit(disp=0).params.iloc[1]
    citl = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(), offset=lp).fit().params.iloc[0]
    return citl, slope

def report(name, y, p):
    citl, slope = calib(y, p)
    print(f"{name:22s} AUC {roc_auc_score(y, p):.3f} | Brier {brier_score_loss(y, p):.4f} | "
          f"관찰 {y.mean():.3f} vs 예측 {p.mean():.3f} | 보정 절편 {citl:+.2f} 기울기 {slope:.2f}")

rand = rng.random(n) < 0.7
m = LogisticRegression(max_iter=1000).fit(df.loc[rand, X], df.died[rand])
report("무작위 30% 시험", df.died[~rand], m.predict_proba(df.loc[~rand, X])[:, 1])
dev, val = df.year <= 2017, df.year >= 2019
m_t = LogisticRegression(max_iter=1000).fit(df.loc[dev, X], df.died[dev])
p_val = m_t.predict_proba(df.loc[val, X])[:, 1]
report("2013–17 개발→2019–20", df.died[val], p_val)

t = pd.DataFrame({"p": p_val, "y": df.died[val].values})
t["decile"] = pd.qcut(t.p, 10, labels=False) + 1
print(t.groupby("decile").agg(예측=("p", "mean"), 관찰=("y", "mean"), n=("y", "size")).round(3).iloc[[0, 4, 9]])

### 예시 2 — 재보정: 숫자는 고치지만 순서는 그대로

2018년 자료(개발에도 검증에도 쓰지 않은 해)로 절편과 기울기만 다시 맞춥니다. 예시 1 다음 셀에 실행합니다.

In [ ]:
rec = df.year == 2018
p_rec = m_t.predict_proba(df.loc[rec, X])[:, 1]
lp_rec = np.log(p_rec / (1 - p_rec))
rc = sm.Logit(df.died[rec], sm.add_constant(lp_rec)).fit(disp=0).params
lp_val = np.log(p_val / (1 - p_val))
p_recal = 1 / (1 + np.exp(-(rc.iloc[0] + rc.iloc[1] * lp_val)))
report("재보정 후 2019–20", df.died[val], p_recal)

### 예시 3 — 누수 변수 하나가 만드는 "놀라운" 성능

사망한 환자의 60%(와 생존 환자의 3%)에게 내려진 가상의 "완화 치료 오더"를 특징으로 넣어 봅니다. 이 오더는 결과가 가까워졌다는 의료진의 판단이 기록된 것이라 예측 시점에는 존재하지 않습니다.

In [ ]:
df["comfort_care_order"] = ((df.died == 1) & (rng.random(n) < 0.6)) | (rng.random(n) < 0.03)
X_leak = X + ["comfort_care_order"]
m_l = LogisticRegression(max_iter=1000).fit(df.loc[dev, X_leak], df.died[dev])
report("누수 변수 포함", df.died[val], m_l.predict_proba(df.loc[val, X_leak])[:, 1])

## 직접 해보기

1. `notebooks/advanced/E/E6_prediction.ipynb`에서 예시 1–3을 실행합니다.
2. 보정 그림을 그립니다. 예시 2 다음 셀에 붙여 넣습니다.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

out = Path("../../../figures/advanced/E"); out.mkdir(parents=True, exist_ok=True)
plt.figure(figsize=(4.5, 4.5))
for name, p in [("before recalibration", p_val), ("after recalibration", p_recal)]:
    g = pd.DataFrame({"p": p, "y": df.died[val].values})
    g["bin"] = pd.qcut(g.p, 10, labels=False)
    s = g.groupby("bin").agg(pred=("p", "mean"), obs=("y", "mean"))
    plt.plot(s.pred, s.obs, "o-", label=name)
plt.plot([0, 0.5], [0, 0.5], "k--", lw=1, label="perfect")
plt.xlabel("mean predicted risk"); plt.ylabel("observed proportion")
plt.title("Temporal validation 2019–20")
plt.legend(); plt.tight_layout()
plt.savefig(out / "E6_calibration.png", dpi=150)
plt.show()